# Predict Customer Churn — Version 5

## Feature Selection with Optuna Tuning
- **Feature Engineering**: V4 features + feature selection (drop bottom 5%)
- **Optuna**: 30 trials XGB, 10 trials LGBM, 30 trials CatBoost
- **Models**: XGBoost + LightGBM + CatBoost (no LogReg)
- **Output**: V14-V16 with rank blend strategy
- **Key Change**: Feature importance filtering before training

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import os, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
from scipy.stats import rankdata

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
N_SEEDS = 10
np.random.seed(SEED)

print('All libraries loaded!')

## 2. Load All Datasets

In [ ]:
# Competition data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Original IBM real data
DATASET = '/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset'
original = pd.read_csv(f'{DATASET}/Original.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')

# Fix TotalCharges in original IBM data
original['TotalCharges'] = pd.to_numeric(
    original['TotalCharges'], errors='coerce'
).fillna(0)

# Add fake id to original
original['id'] = range(900000, 900000 + len(original))

# Combine train + original IBM data
train_combined = pd.concat([train, original], ignore_index=True)
print(f'Combined train: {train_combined.shape} ({len(train):,} competition + {len(original):,} IBM)')

# Smoothed Target Encoding from IBM original
orig_enc = original.copy()
orig_enc['Churn_bin'] = (orig_enc['Churn'] == 'Yes').astype(int)
global_mean = orig_enc['Churn_bin'].mean()
k = 10  # smoothing factor

te_cols = ['Contract', 'InternetService', 'PaymentMethod', 'tenure']
orig_te = {}
for col in te_cols:
    if col in orig_enc.columns:
        stats = orig_enc.groupby(col)['Churn_bin'].agg(['mean', 'count'])
        stats['smoothed'] = (
            (stats['mean'] * stats['count'] + global_mean * k)
            / (stats['count'] + k)
        )
        orig_te[col] = stats['smoothed'].to_dict()
        print(f'Target encoding ready: {col} ({len(stats)} categories)')

# Load previous best submissions for cross-version blend (V9 = 0.91404)
v9_sub = pd.read_csv(f'{DATASET}/V9.csv')
print(f'V9 loaded for cross-version blend: {v9_sub.shape}')

## 3. Feature Engineering

In [ ]:
def preprocess(df, is_train=True, freq_maps=None, fit_freq=False):
    df = df.copy()
    df.drop(columns=['id', 'customerID'], inplace=True, errors='ignore')
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)

    for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    df['gender'] = (df['gender'] == 'Male').astype(int)

    # BASE FEATURES (V3)
    df['AvgMonthlyCharge']  = df['TotalCharges'] / df['tenure'].clip(lower=1)
    df['ChargeRatio']       = df['MonthlyCharges'] / df['AvgMonthlyCharge'].clip(lower=0.01)
    df['IsNewCustomer']     = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']        = (df['tenure'] >= 60).astype(int)
    df['ChargeXTenure']     = df['MonthlyCharges'] * df['tenure']
    df['IsHighValue']       = (df['MonthlyCharges'] > 70).astype(int)

    if 'Contract' in df.columns:
        cr_map = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
        df['ContractRisk']       = df['Contract'].map(cr_map).fillna(2)
        df['TenureContractRisk'] = df['tenure'] * df['ContractRisk']
        df['ChargeContractRisk'] = df['MonthlyCharges'] * df['ContractRisk']
        df['HighRiskFlag']       = (
            (df['ContractRisk'] == 3) & (df['IsNewCustomer'] == 1)
        ).astype(int)

    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    temp_bins = []
    for col in service_cols:
        if col in df.columns and df[col].dtype == object:
            bname = col + '_bin'
            df[bname] = (df[col] == 'Yes').astype(int)
            temp_bins.append(bname)
    df['TotalServices'] = df['PhoneService'].copy()
    for bname in temp_bins:
        df['TotalServices'] += df[bname]
    df['ChargePerService'] = df['MonthlyCharges'] / df['TotalServices'].clip(lower=1)

    # NEW V4 FEATURES
    if 'PaymentMethod' in df.columns:
        df['AutoPay'] = df['PaymentMethod'].str.contains('automatic',case=False,na=False).astype(int)
        df['ElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

    if 'InternetService' in df.columns:
        df['FiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
        df['NoInternet'] = (df['InternetService'] == 'No').astype(int)

    if 'SeniorCitizen' in df.columns and 'FiberOptic' in df.columns:
        df['SeniorXFiber']   = df['SeniorCitizen'] * df['FiberOptic']
        df['SeniorXMonthly'] = df['SeniorCitizen'] * df['MonthlyCharges']

    df['LifetimeValue'] = df['TotalCharges'] / df['MonthlyCharges'].clip(lower=0.01)
    df['ChargeGrowth']  = df['MonthlyCharges'] - df['AvgMonthlyCharge']
    df['tenure_sq']   = df['tenure'] ** 2
    df['monthly_sq']  = df['MonthlyCharges'] ** 2

    df.drop(columns=temp_bins, inplace=True, errors='ignore')

    # TARGET ENCODING FROM IBM DATA
    for col, te_dict in orig_te.items():
        if col in df.columns:
            fallback = float(np.mean(list(te_dict.values())))
            df[f'{col}_te_ibm'] = df[col].map(te_dict).fillna(fallback)

    # ONE-HOT ENCODING
    ohe_cols = [
        'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
    ]
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)

    return df, freq_maps

# Apply preprocessing
train_clean, freq_maps = preprocess(train_combined, is_train=True, fit_freq=True)
test_clean,  _         = preprocess(test, is_train=False, freq_maps=freq_maps)

# Align columns between train and test
feature_cols = [c for c in train_clean.columns if c != 'Churn']
for col in feature_cols:
    if col not in test_clean.columns:
        test_clean[col] = 0
test_clean = test_clean[feature_cols]

print(f'Train clean: {train_clean.shape}')
print(f'Test clean:  {test_clean.shape}')

## 4. Feature Selection — Drop Noisy Features

In [ ]:
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn'])

print(f'Before selection: {X.shape[1]} features')

# Quick XGB to get feature importances (fast, no tuning)
xgb_fs = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda',
    random_state=42,
    verbosity=0
)
xgb_fs.fit(X, y)

importances = pd.Series(xgb_fs.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

# Drop features with zero or near-zero importance (bottom 5%)
threshold    = importances.quantile(0.05)
keep_cols    = importances[importances > threshold].index.tolist()
drop_cols    = importances[importances <= threshold].index.tolist()

print(f'Dropping {len(drop_cols)} low-importance features')
print(f'Keeping  {len(keep_cols)} features')

X          = X[keep_cols]
test_clean = test_clean[keep_cols]

X_arr    = X.values.astype(np.float32)
y_arr    = y.values
test_arr = test_clean.values.astype(np.float32)

# Plot top 30 features
plt.figure(figsize=(10, 8))
importances[keep_cols][:30].plot(kind='barh', color='steelblue')
plt.title('Top 30 Feature Importances (XGB)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_v5.png', dpi=150)
plt.show()

print('Feature selection complete!')

## 5. Setup — X, Y, CV

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
churn_ratio = y.mean()
scale_pos_weight = (1 - churn_ratio) / churn_ratio

print(f'Features: {X.shape[1]}')
print(f'Training rows: {len(X):,}')
print(f'Churn rate: {churn_ratio:.2%}')
print(f'CV folds: {N_FOLDS}')
print(f'Seeds: {N_SEEDS}')

# Pre-loaded best params
best_xgb_params = {
    'n_estimators': 1295,
    'max_depth': 6,
    'learning_rate': 0.02525,
    'subsample': 0.9373,
    'colsample_bytree': 0.5382,
    'min_child_weight': 20,
    'reg_alpha': 0.01357,
    'reg_lambda': 2.8e-05,
    'gamma': 3.79e-07,
}

best_lgbm_params = {
    'n_estimators': 1049,
    'max_depth': 7,
    'learning_rate': 0.02706,
    'num_leaves': 55,
    'min_child_samples': 71,
    'subsample': 0.8215,
    'bagging_freq': 1,
    'colsample_bytree': 0.7317,
    'reg_alpha': 7.34,
    'reg_lambda': 0.000117,
}

v3_cat_params = {
    'iterations': 965,
    'depth': 4,
    'learning_rate': 0.11515,
    'l2_leaf_reg': 0.0032,
    'bagging_temperature': 0.3323,
    'random_strength': 4.563,
    'border_count': 221,
}

print('Pre-loaded best params from V2/V3')

## 6. Optuna Tuning — All 3 Models

In [ ]:
# XGBoost Optuna (30 trials)
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
    }
    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        device='cuda',
        random_state=42,
        verbosity=0
    )
    scores = cross_val_score(model, X_arr, y_arr, cv=cv, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('Running Optuna for XGBoost (30 trials)...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f'XGBoost Best CV: {study_xgb.best_value:.5f}')

# LightGBM Optuna (10 trials)
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'bagging_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    model = LGBMClassifier(
        **params,
        objective='binary',
        metric='auc',
        device='cpu',
        n_jobs=-1,
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    scores = cross_val_score(model, X_arr, y_arr, cv=cv, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('\nRunning Optuna for LightGBM (10 trials)...')
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=10, show_progress_bar=True)
best_lgbm_params = study_lgbm.best_params
print(f'LightGBM Best CV: {study_lgbm.best_value:.5f}')

# CatBoost Optuna (30 trials)
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1500),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
    }
    model = CatBoostClassifier(
        **params,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=0,
        task_type='CPU',
        auto_class_weights='Balanced'
    )
    scores = cross_val_score(model, X_arr, y_arr, cv=cv, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('\nRunning Optuna for CatBoost (30 trials)...')
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=30, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f'CatBoost Best CV: {study_cat.best_value:.5f}')

print('\n=== ALL OPTUNA COMPLETE ===')

## 7. OOF Generation — 3 Models (No LogReg)

In [ ]:
oof_xgb  = np.zeros(len(X))
oof_lgbm = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_xgb_oof  = np.zeros(len(test_clean))
test_lgbm_oof = np.zeros(len(test_clean))
test_cat_oof  = np.zeros(len(test_clean))

print(f'Generating OOF — {N_FOLDS} folds, 3 models...')
print()

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_arr, y_arr)):
    print(f'--- Fold {fold+1}/{N_FOLDS} ---')

    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

    # XGBoost
    xgb_m = XGBClassifier(
        **best_xgb_params,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        device='cuda',
        random_state=42,
        verbosity=0
    )
    xgb_m.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_m.predict_proba(X_val)[:, 1]
    test_xgb_oof += xgb_m.predict_proba(test_arr)[:, 1] / N_FOLDS
    xgb_auc = roc_auc_score(y_val, oof_xgb[val_idx])
    print(f'  XGBoost:  {xgb_auc:.5f}')

    # LightGBM
    lgbm_m = LGBMClassifier(
        **best_lgbm_params,
        objective='binary',
        metric='auc',
        device='cpu',
        n_jobs=-1,
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    lgbm_m.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = lgbm_m.predict_proba(X_val)[:, 1]
    test_lgbm_oof += lgbm_m.predict_proba(test_arr)[:, 1] / N_FOLDS
    lgbm_auc = roc_auc_score(y_val, oof_lgbm[val_idx])
    print(f'  LightGBM: {lgbm_auc:.5f}')

    # CatBoost
    cat_m = CatBoostClassifier(
        **best_cat_params,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=0,
        task_type='CPU',
        auto_class_weights='Balanced'
    )
    cat_m.fit(X_tr, y_tr)
    oof_cat[val_idx] = cat_m.predict_proba(X_val)[:, 1]
    test_cat_oof += cat_m.predict_proba(test_arr)[:, 1] / N_FOLDS
    cat_auc = roc_auc_score(y_val, oof_cat[val_idx])
    print(f'  CatBoost: {cat_auc:.5f}')
    print()

xgb_cv_score  = roc_auc_score(y_arr, oof_xgb)
lgbm_cv_score = roc_auc_score(y_arr, oof_lgbm)
cat_cv_score  = roc_auc_score(y_arr, oof_cat)

print(f'XGBoost:  {xgb_cv_score:.5f}')
print(f'LightGBM: {lgbm_cv_score:.5f}')
print(f'CatBoost: {cat_cv_score:.5f}')
print(f'\nTarget: > 0.91582 (V3 best CV)')

## 8. Final Ensemble & Submissions

In [ ]:
def rank_blend(arrays, weights):
    n      = len(arrays[0])
    ranked = [rankdata(a) / n for a in arrays]
    return np.clip(sum(w * r for w, r in zip(weights, ranked)), 0, 1)

# V14: Rank Blend V5
blend_v14 = rank_blend(
    [test_xgb_oof, test_lgbm_oof, test_cat_oof],
    weights=[0.40, 0.35, 0.25]
)
sub_v14 = pd.DataFrame({'id': test['id'], 'Churn': blend_v14})
sub_v14.to_csv('submission_v14_rank_blend_v5.csv', index=False)
print(f'V14 (Rank Blend V5):      mean={blend_v14.mean():.4f} -> saved!')

# V15: Cross-Version V13 + V14
v13_sub = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V13.csv')
v13_preds = v13_sub.set_index('id').loc[test['id'], 'Churn'].values
blend_v15 = rank_blend([v13_preds, blend_v14], weights=[0.5, 0.5])
sub_v15 = pd.DataFrame({'id': test['id'], 'Churn': blend_v15})
sub_v15.to_csv('submission_v15_cross_v13_v14.csv', index=False)
print(f'V15 (V13+V14 Cross):      mean={blend_v15.mean():.4f} -> saved!')

# V16: 3-Way Cross V9 + V13 + V14
v9_preds = v9_sub.set_index('id').loc[test['id'], 'Churn'].values
blend_v16 = rank_blend([v9_preds, v13_preds, blend_v14], weights=[0.33, 0.34, 0.33])
sub_v16 = pd.DataFrame({'id': test['id'], 'Churn': blend_v16})
sub_v16.to_csv('submission_v16_3way_cross.csv', index=False)
print(f'V16 (V9+V13+V14 3-way):   mean={blend_v16.mean():.4f} -> saved!')

print()
print('Submission order: V14 first -> V15 -> V16')

## 9. Final Summary

In [ ]:
print('=' * 65)
print('VERSION 5 — FINAL SUMMARY')
print('=' * 65)
print(f'Training rows:       {len(X):,} (competition + IBM)')
print(f'Features:            {X.shape[1]} (after importance filtering)')
print(f'Seeds:               {N_SEEDS}')
print(f'Models:              XGBoost + LightGBM + CatBoost (no LogReg)')
print(f'Optuna trials:       30 + 10 + 30 (total 70)')
print()
print(f'XGBoost OOF:         {xgb_cv_score:.5f}')
print(f'LightGBM OOF:        {lgbm_cv_score:.5f}')
print(f'CatBoost OOF:        {cat_cv_score:.5f}')
print()
print('SUBMISSIONS (max 3 today):')
print(f'  V14 -> Rank Blend V5   mean={blend_v14.mean():.4f} <- submit first')
print(f'  V15 -> Cross V13+V14   mean={blend_v15.mean():.4f} <- submit second')
print(f'  V16 -> 3-Way Cross     mean={blend_v16.mean():.4f} <- submit third')
print()
print('STRATEGY:')
print('  1. Submit V14 -> check if feature selection improves LB')
print('  2. Monitor CV-LB gap')
print('  3. Add V14/V15/V16 to dataset for V6 cross-blending')
print('=' * 65)